In [0]:
# spark.conf.set("spark.sql.shufflePartitions", "1")
from pyspark.sql.functions import rand, expr 


df_impressions = (
    spark.readStream
        .format("rate")
        .option("rowsPerSecond", "5")
        .option("num_partitions", "1")
        .load()
        .selectExpr("value AS adId", "timestamp AS impressionTime")
)


df_clicks = (
    spark.readStream
        .format("rate")
        .option("rowsPerSecond", "5")
        .option("num_partitions", "1")
        .load()
        .where((rand() * 100).cast("integer") < 10)
        .selectExpr("(value - 50) AS adId", "timestamp AS clickTime")
        .where("adId > 0")
)

# State Grows indefinitely
# df_impressions.join(df_clicks)
df_impressions_withWatermark = (
    df_impressions.selectExpr("adId AS impressionAdId", "impressionTime")
    .withWatermark("impressionTime", '10 seconds')
)

df_clicks_withWatermark = (
    df_clicks.selectExpr("adId AS clickAdId", "clickTime")
    .withWatermark("clickTime", '20 seconds')
)


df_impressions_withWatermark.join(
    df_clicks_withWatermark,
    expr("""
        clickAdId = impressionAdId AND
        clickTime >= impressionTime AND
        clickTime <= impressionTime + interval 1 minutes
    """)
)
